# AVA-VLA FFT · canonical D · Suite-balanced · CU 예산

새 Run ID에서 실행합니다. 공유 소스·TAR·T0와 기존 노트북을 수정하지 않습니다.
Train=Suite-balanced(4 suite 각 25%, 부족한 suite replacement), Validation=Episode입니다.
Test는 무결성 검사 외에는 사용하지 않습니다. AVA 학습은 기존 **zero-context FFT**이며
temporal 학습이나 원 논문 재현을 의미하지 않습니다. Cache/T1/Student/양자화는 포함하지 않습니다.

기본값은 준비 및 학습을 실행하지 않습니다. ① 설정/Drive 점검 → ② PREPARE_INPUTS=True로
로컬 검증·Python 3.11 구성 → ③ CU를 기록하고 RUN_PILOT=True로 4-step 실행 →
④ 파일럿 후 CU와 현재 CU를 입력 → ⑤ 예산을 읽고 RUN_MAIN_FFT=True로 실행합니다.
파일럿과 본 학습은 별도 프로세스이며 본 학습은 항상 T0 또는 명시한 본 학습 checkpoint에서 시작합니다.

CU 공식 잔액 API를 사용하지 않습니다. 사용자가 Colab UI의 잔액을 입력합니다.
5 CU 잔여는 실측 기반 추정이며 보장되지 않습니다. CU 소모율이 없으면 명시적 시간 상한이 필요하고,
그 모드에서는 5 CU 목표를 제어할 수 없음을 보고합니다.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, math, os, shutil, subprocess, sys, time, uuid
SESSION_STARTED_UNIX = globals().get('SESSION_STARTED_UNIX', time.time())
PROJECT_ROOT = Path('/content/drive/.shortcut-targets-by-id/1jbXOZooUbe-SM9oUrKmNac_4Dfthkgs0/VLADE')
OUTPUT_ROOT = Path('/content/vlade_runs/ava_fft_suite_balanced_seed42')
RUN_ID = globals().get('RUN_ID', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
# 재시작 후에는 기존 Run ID와 RESUME_CHECKPOINT를 명시합니다. 자동 latest 선택 없음.
RESUME_CHECKPOINT = None
PREPARE_INPUTS = False
PREPARATION_CACHE = OUTPUT_ROOT / 'verified_inputs'
LOCAL_VALIDATION_RECEIPTS = Path('/content/vlade_verified_input_receipts')
# Read-only Drive ID/version checks enable cross-runtime hash reuse. No CU API is assumed.
AUTHENTICATE_DRIVE_CACHE = False
RUN_PILOT = False
RUN_MAIN_FFT = False
SAVE_STEPS = 2000
SAVE_OPTIMIZER_STATE = True
AUTO_UNASSIGN_RUNTIME = True
CURRENT_COMPUTE_UNITS = None  # 본 학습 직전 Colab UI에서 갱신
TARGET_REMAINING_CU = 5.0
CU_BEFORE_PILOT = None
CU_AFTER_PILOT = None
# 공식적으로 확인 가능한 세션 종료 Unix 시각을 아는 경우만 입력. 고정 8시간 가정 없음.
SESSION_DEADLINE_UNIX = None
EXPLICIT_MAIN_TIME_LIMIT_SECONDS = None  # CU 실측 불가 시 필수인 사용자 시간 상한
PILOT_TIME_LIMIT_SECONDS = None  # 4-step도 명시적 상한 필요; 모델 로딩/저장/재로딩 포함
# Checkpoint space is checked on the runtime SSD.
TAR_PATH = 'hfasset://prepared_d_seed42'
EXPECTED_TAR_BYTES = 11_371_123_200
EXPECTED_TAR_SHA256 = None  # 독립적으로 검증된 Manifest에서만 채웁니다. 실측값을 기대값으로 복사 금지
SUITE_MAP_ASSET = 'suite_map_seed42'
MAX_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
RUN_DIR = OUTPUT_ROOT / RUN_ID
LOCAL = Path('/content') / ('vlade-suite-' + RUN_ID)
CODE = LOCAL / 'code'
print(json.dumps({'GPU/VRAM': '다음 셀에서 확인', 'project': str(PROJECT_ROOT),
 'TAR': str(TAR_PATH), 'teacher': 'hfasset://ava_t0',
 'dataset': 'LIBERO+ canonical D 80/10/10 seed42', 'episodes': [11477,1428,1442],
 'train_queries':228908, 'train_sampling':'suite_balanced', 'validation_sampling':'episode',
 'suite_target':[.25,.25,.25,.25], 'current_CU':CURRENT_COMPUTE_UNITS,
 'target_remaining_CU':TARGET_REMAINING_CU, 'measured_CU_rate':None, 'possible_steps':None,
 'estimated_runtime_seconds':None, 'checkpoint_bytes':None, 'save_steps':SAVE_STEPS,
 'safe_reserve_seconds':None, 'RUN_PILOT':RUN_PILOT, 'RUN_MAIN_FFT':RUN_MAIN_FFT,
 'session_deadline_unix':SESSION_DEADLINE_UNIX,'explicit_main_seconds':EXPLICIT_MAIN_TIME_LIMIT_SECONDS,
 'tar_expected_sha256':EXPECTED_TAR_SHA256,'tar_integrity':'미검증',
 'actual_training_started':False, 'output':str(RUN_DIR)}, indent=2, ensure_ascii=False))

## Drive와 실제 경로
TAR ID `17Nim7jPu0dC-wKNkNjAtPyUJAre4QYnM`, 11,371,123,200 bytes를 조회했습니다.
실제 부모는 `data/splits`입니다. 중첩된 공유 압축 해제 폴더는 사용하지 않습니다.
TAR 기대 SHA256은 아직 확보하지 못했습니다. 실측 SHA256 기록만으로 검증 완료라고 하지 않습니다.
대신 로컬 보관 canonical D의 고정 manifest/split/normalization 해시 및 모든 payload를 대조하는
동등 무결성 경로를 사용합니다. 어느 하나라도 불일치하면 학습을 중단합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     text=True, capture_output=True)
print('GPU / VRAM:', gpu.stdout.strip() or gpu.stderr.strip())
for path in [PROJECT_ROOT, TAR_PATH, PROJECT_ROOT / 'checkpoints/teacher/avavla-libero-4in1']:
    print('존재 여부:', path.exists(), path)
print('Suite map 후보:', [(str(p),p.is_file()) for p in SUITE_MAP_CANDIDATES])
print('학습 실행:', False, '| 준비 플래그:', PREPARE_INPUTS)

## 검증된 파일 목록만 Colab 로컬로 준비
공유 source manifest SHA를 먼저 검증합니다. 최근 Suite-balanced 변경 두 파일은
2026-11-04에 Drive에서 읽은 SHA256으로 별도 고정합니다. 기존 source manifest는 수정하지 않습니다.
신규 어댑터는 이 노트북에 내장되어 로컬에만 쓰이며 아래 embedded SHA로 검증합니다.
기존 `fft.py`, `prepared_fft.py`, `episode_data.py`를 덮어쓰지 않습니다.

In [ ]:
READY = False
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
def control_write(path, value):
    path.parent.mkdir(parents=True,exist_ok=True)
    raw = json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False,default=str)
    temp = path.with_suffix(path.suffix+'.tmp')
    with temp.open('w') as f:
        f.write(raw); f.flush(); os.fsync(f.fileno())
    temp.replace(path)
    assert json.loads(path.read_text()) == json.loads(raw)
def run(args, env=None):
    command=[str(x) for x in args]
    started=time.monotonic()
    label=Path(command[0]).name+' '+(Path(command[1]).name if len(command)>1 else '')
    print('[명령 시작]',label,flush=True)
    output=[]
    process=subprocess.Popen(command,cwd=CODE,env=env or ENV,text=True,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True);output.append(line)
    returncode=process.wait();text=''.join(output)
    print('[명령 종료]',label,f'{time.monotonic()-started:.1f}s',flush=True)
    if returncode:
        if (RUN_DIR/'logs').is_dir():
            with (RUN_DIR/'logs/errors.log').open('a') as stream:stream.write(text)
        raise RuntimeError(f'하위 프로세스 실패 (exit={returncode}): {label}')
    return subprocess.CompletedProcess(command,returncode,text,'')
SOURCE_MANIFEST_SHA = 'bef4ecfba258a81465a103ae88096ea799bb25954063f9dd0d82339758c96f7f'
SOURCE_OVERRIDES = {}
if PREPARE_INPUTS:
    PREPARATION_STARTED_MONOTONIC=time.monotonic()
    print('[준비 시작] source SHA 검사 및 Colab 로컬 소스',flush=True)
    if LOCAL.exists() and not CODE.is_dir():raise RuntimeError('기존 local 경로의 소스 구조를 확인하세요.')
    if RUN_DIR.exists() and RESUME_CHECKPOINT is None:
        raise FileExistsError('기존 Run을 덮어쓰지 않습니다. 새 Run ID 또는 명시적 Resume를 사용하세요.')
    manifest_path = PROJECT_ROOT/'experiments/manifests/colab_source.json'
    if digest(manifest_path) != SOURCE_MANIFEST_SHA: raise ValueError('공유 source manifest SHA 불일치')
    source_manifest = json.loads(manifest_path.read_text())
    inventory = {**source_manifest['files_sha256'], **SOURCE_OVERRIDES}
    selected = {p:h for p,h in inventory.items() if p.startswith(('src/','scripts/',
        'configs/','third_party/ava_vla/','experiments/manifests/'))}
    CODE.mkdir(parents=True,exist_ok=True)
    for rel, expected in selected.items():
        relative = Path(rel)
        if relative.is_absolute() or '..' in relative.parts or '\\' in rel or ':' in rel:
            raise ValueError('잘못된 source 경로: '+rel)
        source, target = PROJECT_ROOT/rel, CODE/rel
        if not source.resolve().is_relative_to(PROJECT_ROOT.resolve()): raise ValueError(rel)
        if digest(source) != expected: raise ValueError('공유 소스 SHA 불일치: '+rel)
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():
            if digest(target)!=expected:raise ValueError('기존 local 소스 SHA 불일치: '+rel)
        else:shutil.copyfile(source,target)
        if digest(target) != expected: raise ValueError('로컬 복사 SHA 불일치: '+rel)
    target = CODE/'experiments/manifests/colab_source.json'
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_bytes(manifest_path.read_bytes())
    for folder in ['configs','pilot4','logs','manifests','reports']:
        (RUN_DIR/folder).mkdir(parents=True,exist_ok=True)
    ENV = os.environ.copy()
    ENV.update(PYTHONPATH=str(CODE/'src'),PYTHONDONTWRITEBYTECODE='1',
      TOKENIZERS_PARALLELISM='false',WANDB_MODE='disabled',TF_CPP_MIN_LOG_LEVEL='2',
      HF_HOME=str(LOCAL/'hf_cache'),UV_CACHE_DIR=str(LOCAL/'uv_cache'),
      TMPDIR=str(LOCAL/'tmp'),XDG_CACHE_HOME=str(LOCAL/'cache'))
    (LOCAL/'tmp').mkdir(exist_ok=True)
    # Clone labels remain provenance labels, not proof of a clean git checkout.
    control_write(RUN_DIR/'manifests/source_manifest.json',{
      'source_manifest_sha256':SOURCE_MANIFEST_SHA,'source_revisions':source_manifest['source_revisions'],
      'files_sha256':selected,'overrides':SOURCE_OVERRIDES,'local_only_hooks':True})
    print('검증 후 준비한 로컬 소스:',CODE)
    print('[준비 완료] source SHA / local source',f'{time.monotonic()-PREPARATION_STARTED_MONOTONIC:.1f}s',flush=True)
else:
    print('준비 대기: PREPARE_INPUTS=False. 학습 실행 없음.')
if PREPARE_INPUTS:
    # HF inputs are cached on the runtime SSD; Drive retains code and experiment records.
    ASSET_CACHE = Path('/content/vlade_hf_assets')
    ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
    RUNS_ROOT = Path('/content/vlade_runs')
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
               VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
               HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
    os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})
    _HF_REGISTRY_READY = False

    def hf_input(key, python=None, *, fetch=True):
        global _HF_REGISTRY_READY
        from vlade.storage.assets import asset_record, cache_path
        if fetch and not _HF_REGISTRY_READY:
            if python is None:
                raise ValueError('Select the prepared Python environment before fetching HF assets')
            registry = ASSET_CACHE / 'registry.json'
            run([python, 'scripts/data/hf_assets.py', 'refresh-registry', '--output', registry], env=ENV)
            ENV['VLADE_ASSET_REGISTRY'] = str(registry)
            os.environ['VLADE_ASSET_REGISTRY'] = str(registry)
            _HF_REGISTRY_READY = True
        record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
        target = cache_path(key, record, ASSET_CACHE)
        if fetch:
            if python is None:
                raise ValueError('Select the prepared CPU/model Python environment before fetching HF assets')
            run([python, 'scripts/data/hf_assets.py', 'fetch', key,
                 '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
        return target

    def hf_path(value, python=None, *, fetch=True):
        value = str(value)
        return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch) if value.startswith('hfasset:') else Path(value)
    sys.path.insert(0,str(CODE/'src'))


In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
ADAPTER_SOURCE = '"""Colab-only orchestration around the existing FFT and suite-balanced loader.\n\nNo model or optimizer loop is replaced. Hooks live in the child process only;\nshared sources, datasets and pretrained checkpoints remain read-only.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport tarfile\nimport time\nimport traceback\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\nfrom typing import Any\n\n\nclass _SavedBudgetStop(Exception):\n    """Internal control flow caught inside the loop hook, never by fft.train."""\n\n\ndef runtime_api_report(root: Path) -> dict:\n    """Inspect the selected source tree instead of assuming optional budget APIs."""\n    training = root / "src/vlade/training"\n    fft_tree = ast.parse((training / "fft.py").read_text(encoding="utf-8"))\n    loop_tree = ast.parse((training / "loop.py").read_text(encoding="utf-8"))\n    train = next(node for node in fft_tree.body if isinstance(node, ast.FunctionDef) and node.name == "train")\n    imports = [node for node in ast.walk(train) if isinstance(node, ast.ImportFrom)]\n    # A module-global run_loop alias cannot be intercepted by changing loop.run_loop later.\n    if not any(node.module == "vlade.training.loop" and any(a.name == "run_loop" for a in node.names)\n               for node in imports):\n        raise ValueError("Unsupported FFT API: train must bind loop.run_loop at invocation")\n    run_loop = next(node for node in loop_tree.body if isinstance(node, ast.FunctionDef) and node.name == "run_loop")\n    if "on_step" not in {arg.arg for arg in run_loop.args.kwonlyargs}:\n        raise ValueError("Unsupported loop API: on_step callback missing")\n    budget = training / "compute_budget.py"\n    api = {"exists": budget.is_file(), "adapter_uses_native_budget": False}\n    if budget.is_file():\n        tree = ast.parse(budget.read_text(encoding="utf-8"))\n        names = {node.name for node in tree.body if isinstance(node, (ast.FunctionDef, ast.ClassDef))}\n        api.update(sha256=sha256(budget), names=sorted(names))\n        stop = next((node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "should_stop"), None)\n        api["known_api"] = bool("BudgetExhausted" in names and stop and\n            [arg.arg for arg in stop.args.args] == ["remaining_seconds", "save_reserve_seconds", "next_step_seconds"])\n    elif any(node.module == "vlade.training.compute_budget" for node in imports):\n        raise ValueError("Selected fft.train imports a missing compute_budget.py; source snapshot is incomplete")\n    return {"compute_budget": api, "termination": "adapter_catches_saved_stop_inside_loop_hook",\n            "timing": "adapter_monotonic_with_device_synchronization"}\n\n\ndef validate_suite_resume(cfg: Any, suite_map: Path, fingerprint: dict,\n                          verify_checkpoint: Any) -> dict:\n    """Reject incompatible cursor/state before the original trainer restores RNG."""\n    import torch\n    verify_checkpoint(cfg.resume, accumulation=cfg.grad_accumulation_steps)\n    sampling = read(cfg.resume / "suite_sampling.json")\n    payload = torch.load(cfg.resume / "trainer_state.pt", map_location="cpu", weights_only=True)\n    state = payload["loop"]\n    step = read(cfg.resume / "vlade_fft.json")["step"]\n    if (sampling.get("sampling") != "suite_balanced" or sampling.get("validation_sampling") != "episode"\n            or sampling.get("seed") != cfg.seed or sampling.get("suite_map_sha256") != sha256(suite_map)\n            or sampling.get("fingerprint") != fingerprint\n            or sampling.get("completed_optimizer_step") != step\n            or sampling.get("query_cursor") != step * cfg.grad_accumulation_steps * cfg.batch_size\n            or state.get("step") != step or state.get("microsteps") != step * cfg.grad_accumulation_steps\n            or any(key not in payload for key in ("optimizer", "scheduler", "rng"))\n            or any(key not in payload["rng"] for key in ("python", "numpy", "torch", "cuda"))):\n        raise ValueError("Resume suite cursor, fingerprint, optimizer boundary or RNG state mismatch")\n    return sampling\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef read(path: Path) -> Any:\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef write(path: Path, value: Any) -> None:\n    """Write and read back a small control file; never overwrite checkpoints."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(value, indent=2, default=str, allow_nan=False)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8") as stream:\n        stream.write(payload)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n    if read(path) != json.loads(payload):\n        raise IOError(f"Control file readback failed: {path}")\n\n\ndef extract_archive(archive: Path, expected_sha256: str | None, destination: Path) -> Path:\n    """Validate an explicitly selected TAR, then extract to a fresh local folder."""\n    if destination.exists() or (expected_sha256 is not None and len(expected_sha256) != 64):\n        raise ValueError("Use a new local destination and a valid expected SHA256, if available")\n    if expected_sha256 is not None and sha256(archive) != expected_sha256:\n        raise ValueError("TAR SHA256 mismatch")\n    with tarfile.open(archive, "r:*") as stream:\n        members = stream.getmembers()\n        total = 0\n        seen = set()\n        for member in members:\n            target = (destination / member.name).resolve()\n            if (not target.is_relative_to(destination.resolve()) or\n                    not (member.isfile() or member.isdir()) or target in seen):\n                raise ValueError(f"Unsafe/duplicate TAR member: {member.name}")\n            seen.add(target)\n            total += member.size\n        if shutil.disk_usage(destination.parent).free < total * 1.1:\n            raise ValueError("Insufficient Colab local extraction space")\n        destination.mkdir()\n        stream.extractall(destination, members=members, filter="data")\n    candidates = [p.parent for p in destination.rglob("manifest.json")\n                  if (p.parent / "splits.json").is_file() and (p.parent / "normalization.json").is_file()]\n    if len(candidates) != 1:\n        raise ValueError("Expected exactly one prepared dataset inside the TAR")\n    return candidates[0]\n\n\ndef verify_data(root: Path, suite_map: Path) -> dict:\n    """Check the canonical D identity and every declared payload, including sealed test files."""\n    from vlade.data.episode_data import EpisodeData, query_count\n    expected = {"manifest.json": "9a09c51dac170cb7452e0f3cd33f563e73429d6fe8f7a880daa8568ec2cf981b",\n                "splits.json": "379376e5cfb19d0b9d0a5bdd548e01e4c658f09de3fdd1bff178b3aad7ed9900",\n                "normalization.json": "e1dbf29ea83cc2e0fbcb90e68a7ad15660eb210baa5a333d20bd16c80d275af3"}\n    if any(sha256(root / name) != digest for name, digest in expected.items()):\n        raise ValueError("Extracted data differs from the recorded canonical D prepared manifest")\n    data = EpisodeData(root)\n    if (data.manifest.get("counts") != {"train": 11477, "validation": 1428, "test": 1442}\n            or data.manifest.get("seed") != 42 or query_count(data) != 228908\n            or data.manifest.get("split_policy") != "canonical_D_content_group_80_10_10_seed_42"\n            or data.manifest.get("canonical_manifest_sha256") !=\n            "487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80"\n            or not data.stats.get("train_only") or data.stats.get("num_trajectories") != 11477):\n        raise ValueError("Dataset does not match the confirmed D split")\n    for name, dim in (("action", 7), ("proprio", 8)):\n        if any(len(data.stats[name][key]) != dim for key in ("q01", "q99", "mask")):\n            raise ValueError("BOUNDS_Q99 statistics dimension mismatch")\n    for entry in data.entries:\n        folder = root / "episodes" / entry["episode_id"]\n        for name, expected in entry["files"].items():\n            path = folder / name\n            if not path.resolve().is_relative_to(folder.resolve()) or path.is_symlink() or sha256(path) != expected:\n                raise ValueError(f"Dataset payload hash mismatch: {path}")\n    # Reuse the exact suite mapping validation used by the existing sampler.\n    from vlade.data.episode_data import _read_suite_mapping, _episode_suite\n    by_episode, by_instruction = _read_suite_mapping(data, suite_map)\n    counts = dict.fromkeys(("spatial", "goal", "object", "libero_10"), 0)\n    for entry in data.entries:\n        if entry["split"] == "train":\n            queries = read(root / "episodes" / entry["episode_id"] / "queries.json")\n            counts[_episode_suite(entry, queries, by_episode, by_instruction)] += entry["queries"]\n    if not all(counts.values()):\n        raise ValueError("An empty suite cannot be balanced")\n    return {"counts": data.manifest["counts"], "train_queries": query_count(data),\n            "suite_unique_queries": counts, "target_proportions": {k: .25 for k in counts},\n            "sampling": "suite_balanced", "validation_sampling": "episode", "seed": 42,\n            "replacement": "underrepresented_suites", "normalization": "train_only_BOUNDS_Q99",\n            "action_chunk": 8, "action_dim": 7, "proprio_dim": 8,\n            "files_sha256": {n: sha256(root / n) for n in ("manifest.json", "splits.json", "normalization.json")},\n            "suite_map_sha256": sha256(suite_map), "payloads_verified": True,\n            "test_usage": "integrity_only_never_training_or_selection"}\n\n\ndef plan_budget(*, profile: dict, current_cu: float | None, target_cu: float,\n                cu_before: float | None, cu_after: float | None, cu_elapsed: float,\n                session_seconds: float | None, explicit_seconds: float | None,\n                free_bytes: int, checkpoint_bytes: int, remaining_steps: int,\n                save_steps: int = 2000, margin: float = 1.35, resume_step: int = 0) -> dict:\n    """Budget all remaining saves and keep checkpoint storage append-only."""\n    def positive(value: Any) -> bool:\n        return type(value) in (int, float) and math.isfinite(value) and value > 0\n    if (not positive(target_cu) or not positive(cu_elapsed) or not positive(margin) or margin < 1\n            or any(not positive(profile[k]) for k in\n                   ("step_seconds", "checkpoint_seconds", "startup_seconds", "reload_seconds"))\n            or type(resume_step) is not int or resume_step < 0\n            or any(type(v) is not int or v < 1 for v in\n                   (free_bytes, checkpoint_bytes, remaining_steps, save_steps))):\n        raise ValueError("Invalid pilot, storage, or budget input")\n    rate = None\n    if cu_before is not None or cu_after is not None:\n        if not positive(cu_before) or not positive(cu_after) or cu_before <= cu_after:\n            raise ValueError("CU readings must show positive measured consumption; extend measurement if rounded")\n        rate = (cu_before - cu_after) / cu_elapsed\n    limits = []\n    if rate is not None:\n        if not positive(current_cu) or current_cu <= target_cu:\n            raise ValueError("Enter current CU above the target reserve")\n        limits.append((current_cu - target_cu) / (rate * margin))\n    elif explicit_seconds is None:\n        raise ValueError("No measured CU rate: enter an explicit time cap; no rate is assumed")\n    for value in (session_seconds, explicit_seconds):\n        if value is not None:\n            if not positive(value):\n                raise ValueError("Time caps must be positive and unexpired")\n            limits.append(value)\n    seconds = min(limits)\n    validation_seconds = profile.get("validation_seconds")\n    if not positive(validation_seconds):\n        raise ValueError("Invalid validation measurement")\n    reserve = max(300., margin * (profile["checkpoint_seconds"] + profile["reload_seconds"] + validation_seconds))\n    startup = profile["startup_seconds"] * margin\n    slot = math.ceil(checkpoint_bytes * margin)\n    # One extra slot covers an incomplete save without deleting earlier checkpoints.\n    slots = free_bytes // slot - 1\n    def save_count(steps: int) -> int:\n        final_step = resume_step + steps\n        return final_step // save_steps - resume_step // save_steps + int(final_step % save_steps != 0)\n    def cost(steps: int) -> float:\n        saves = save_count(steps)\n        return (startup + steps * profile["step_seconds"] * margin\n                + saves * (profile["checkpoint_seconds"] + validation_seconds) * margin + reserve)\n    lo, hi = 0, remaining_steps\n    while lo < hi:\n        middle = (lo + hi + 1) // 2\n        if save_count(middle) <= slots and cost(middle) <= seconds:\n            lo = middle\n        else:\n            hi = middle - 1\n    if lo < 1:\n        raise ValueError("Insufficient measured time/CU or Drive quota for a resumable FFT step")\n    return {"steps_this_session": lo, "remaining_training_steps": remaining_steps,\n            "budget_limited": lo < remaining_steps, "wall_seconds": seconds,\n            "save_steps": save_steps, "save_reserve_seconds": reserve,\n            "measured_cu_per_second": rate, "estimated_cu_per_step": rate * profile["step_seconds"] if rate else None,\n            "current_cu": current_cu, "target_remaining_cu": target_cu,\n            "cu_limit_is_estimated": True, "cu_target_supported": rate is not None,\n            "safety_factor": margin, "estimated_main_seconds": cost(lo),\n            "estimated_full_training_seconds": cost(remaining_steps),\n            "estimated_checkpoint_bytes": checkpoint_bytes, "free_drive_bytes": free_bytes,\n            "planned_checkpoint_count": save_count(lo), "profile": profile,\n            "resume_step": resume_step}\n\n\ndef install_hooks(suite_map: Path) -> dict:\n    """Bind the existing loader and strengthen fingerprints in this process only."""\n    import vlade.training.fft as fft\n    import vlade.training.prepared_fft as prepared\n    import vlade.training.prepared_fft_suite_balanced as balanced\n    from types import SimpleNamespace\n    original = prepared.make_loader\n    fingerprint = fft._code_fingerprint\n    def make_loader(cfg: Any, processor: Any, *, train: bool = True) -> Any:\n        values = asdict(cfg)\n        values["suite_map_path"] = suite_map\n        return balanced.make_loader(SimpleNamespace(**values), processor, train=train)\n    def code_fingerprint() -> dict:\n        result = fingerprint()\n        for path in (Path(__file__), Path(balanced.__file__)):\n            result[path.name] = sha256(path)\n        result["suite_map.json"] = sha256(suite_map)\n        # Include local upstream and patched installed Transformers code, not just a git label.\n        import transformers.models.llama.modeling_llama as llama\n        result["installed_modeling_llama.py"] = sha256(Path(llama.__file__))\n        upstream = fft.REPO_ROOT / "third_party/ava_vla"\n        result["upstream_tree"] = hashlib.sha256(json.dumps({\n            p.relative_to(upstream).as_posix(): sha256(p) for p in sorted(upstream.rglob("*.py"))\n            if "__pycache__" not in p.parts}, sort_keys=True).encode()).hexdigest()\n        teacher = fft.REPO_ROOT.parent / "teacher"\n        if teacher.is_dir():\n            result["teacher_metadata"] = hashlib.sha256(json.dumps({\n                p.name: sha256(p) for p in sorted(teacher.iterdir())\n                if p.is_file() and p.suffix not in {".pt", ".safetensors"}}, sort_keys=True).encode()).hexdigest()\n        return result\n    prepared.make_loader = make_loader\n    fft._code_fingerprint = code_fingerprint\n    return {"previous_loader": original, "fingerprint": code_fingerprint()}\n\n\ndef execute(settings: dict) -> dict:\n    """Run one explicitly selected phase, preserving the existing FFT loop."""\n    import torch\n    import vlade.training.fft as fft\n    import vlade.training.loop as loop\n    import vlade.training.run_control as control\n    runtime_api = runtime_api_report(fft.REPO_ROOT)\n    suite_map = Path(settings["suite_map"])\n    install_hooks(suite_map)\n    cfg = fft.FFTConfig.from_json(Path(settings["config"]), "ava_vla")\n    cfg = replace(cfg, resume=Path(settings["resume"]) if settings.get("resume") else None)\n    phase = settings["phase"]\n    if phase not in {"pilot4", "main"} or not settings.get("authorized"):\n        raise ValueError("Explicit phase execution flag required")\n    if cfg.seed != 42 or cfg.image_aug or not cfg.save_optimizer_state:\n        raise ValueError("Require seed=42, image_aug=false and optimizer state saving")\n    if phase == "pilot4" and (cfg.max_steps != 4 or cfg.resume):\n        raise ValueError("Pilot must start independently from T0 and run four optimizer steps")\n    if phase == "main" and cfg.save_steps != 2000:\n        raise ValueError("Main checkpoint interval must be 2000")\n    run_root = Path(settings["run_root"])\n    started = time.time()\n    started_monotonic = time.monotonic()\n    deadline = float(settings["deadline_unix"])\n    reserve = float(settings["reserve_seconds"])\n    stop_step = int(settings["stop_step"])\n    if not math.isfinite(deadline) or not math.isfinite(reserve) or reserve <= 0 or deadline <= started + reserve:\n        raise ValueError("Expired/invalid execution budget")\n    deadline_monotonic = started_monotonic + deadline - started\n    # Only the adapter owns budget termination. Older FFT catches every Exception as failure;\n    # newer FFT has a different optional budget module. Never send our stop into either path.\n    budget_env = {key: os.environ.get(key) for key in\n                  ("VLADE_TRAIN_DEADLINE_UNIX", "VLADE_DEADLINE_UNIX", "VLADE_SAVE_RESERVE_SECONDS", "VLADE_HUB_RUN_ID")}\n    resume_step = 0\n    if cfg.resume:\n        if not cfg.resume.resolve().is_relative_to((run_root / "checkpoints").resolve()):\n            raise ValueError("Resume must explicitly name this run\'s main checkpoint")\n        resume_step = fft.validate_resume(cfg)["step"]\n        validate_suite_resume(cfg, suite_map, fft._code_fingerprint(), control.verify_checkpoint)\n    if not resume_step < stop_step <= cfg.max_steps:\n        raise ValueError("Invalid session target optimizer step")\n    context: dict = {}\n    saves: list[dict] = []\n    times: list[float] = []\n    evaluations: list[float] = []\n    budget_stopped = False\n    old_model, old_loader = fft._make_model_and_components, fft._make_loader\n    old_save, old_loop, old_seal = fft.save_checkpoint, loop.run_loop, control.seal_checkpoint\n    old_evaluate = loop.evaluate\n    def synchronize() -> None:\n        if str(cfg.device).startswith("cuda"):\n            torch.cuda.synchronize(cfg.device)\n    def evaluate_hook(*args: Any, **kwargs: Any) -> Any:\n        synchronize()\n        before = time.monotonic()\n        try:\n            return old_evaluate(*args, **kwargs)\n        finally:\n            synchronize()\n            elapsed = time.monotonic() - before\n            context.setdefault("validation_timings", []).append(elapsed)\n            if context.get("step_metrics") is not None:\n                context["step_metrics"]["validation_seconds"] = elapsed\n    def model_hook(*args: Any, **kwargs: Any) -> Any:\n        value = old_model(*args, **kwargs)\n        context["model"] = value\n        return value\n    def loader_hook(*args: Any, **kwargs: Any) -> Any:\n        value = old_loader(*args, **kwargs)\n        if kwargs.get("train", True):\n            context["dataset"] = value[1]\n        return value\n    def seal_hook(directory: Path, step: int, microsteps: int) -> dict:\n        for name in ("config.json", "trainer_state.pt", "vlade_fft.json"):\n            if not (directory / name).is_file() or (directory / name).stat().st_size == 0:\n                raise ValueError(f"Cannot publish an incomplete training checkpoint: {name}")\n        write(directory / "suite_sampling.json", {\n            "sampling": "suite_balanced", "validation_sampling": "episode", "seed": 42,\n            "target_proportions": {k: .25 for k in ("spatial", "goal", "object", "libero_10")},\n            "suite_map_sha256": sha256(suite_map), "completed_optimizer_step": step,\n            "query_cursor": step * cfg.grad_accumulation_steps * cfg.batch_size,\n            "cursor_method": "deterministic_replay_skip_num_workers_0_no_image_augmentation",\n            "bitwise_gpu_reproduction_guaranteed": False, "fingerprint": fft._code_fingerprint()})\n        return old_seal(directory, step, microsteps)\n    def save_hook(*args: Any, **kwargs: Any) -> Path:\n        if kwargs.get("hub") is not None:\n            context["hub"] = kwargs["hub"]\n        if settings.get("free_bytes_at_start") is not None:\n            used = sum(p.stat().st_size for p in run_root.rglob("*") if p.is_file())\n            if settings["free_bytes_at_start"] - used < settings["checkpoint_bytes"] * 1.35:\n                raise RuntimeError("Insufficient user-confirmed Drive quota for another checkpoint")\n        before = time.monotonic()\n        path = old_save(*args, **kwargs)\n        control.verify_checkpoint(path, accumulation=cfg.grad_accumulation_steps)\n        elapsed = time.monotonic() - before\n        saves.append({"step": read(path / "vlade_fft.json")["step"], "path": str(path),\n                      "save_verify_seconds": elapsed,\n                      "bytes": sum(p.stat().st_size for p in path.rglob("*") if p.is_file())})\n        return path\n    def loop_hook(**kwargs: Any) -> Any:\n        nonlocal budget_stopped\n        state = kwargs["state"]\n        original_step = kwargs["on_step"]\n        original_batches = kwargs["batches"]\n        max_step_seconds = max(float(settings.get("step_seconds", 0)),\n                               float(settings.get("first_step_seconds", 0)))\n        def save_boundary() -> None:\n            if state.microsteps != state.step * cfg.grad_accumulation_steps:\n                raise ValueError("Cannot checkpoint a partial accumulation window")\n            if state.step < 1:\n                raise RuntimeError("Budget expired before the first optimizer step; no checkpoint saved")\n            path = cfg.output_dir / f"step-{state.step:07d}"\n            if path.exists():\n                control.verify_checkpoint(path, accumulation=cfg.grad_accumulation_steps)\n            else:\n                model, processor, head, proprio, generator = context["model"]\n                from vlade.training.hub import HubCheckpoints\n                hub = context.get("hub")\n                if hub is None:\n                    hub = HubCheckpoints(cfg.output_dir, resume=cfg.resume, metric="val_l1")\n                    context["hub"] = hub\n                save_hook(cfg, state.step, model, processor, head, proprio, generator,\n                          kwargs["optimizer"], context["dataset"], scheduler=kwargs["scheduler"], loop_state=state,\n                          hub=hub, metrics=context.get("last_metrics", {}))\n                hub.publish(path, final=state.step == cfg.max_steps)\n        def guarded_batches() -> Any:\n            # Check before fetching the first microbatch of each accumulation window.\n            iterator = iter(original_batches)\n            micro = 0\n            while True:\n                dynamic_save = max([reserve] + [1.35 * s["save_verify_seconds"] + 300 for s in saves])\n                if micro % cfg.grad_accumulation_steps == 0:\n                    if (state.step >= stop_step or\n                            time.monotonic() + dynamic_save + max_step_seconds * 1.35 >= deadline_monotonic):\n                        save_boundary()\n                        raise _SavedBudgetStop("Saved before the next accumulation window")\n                    synchronize()\n                    context["step_started"] = time.monotonic()\n                    context.setdefault("startup_seconds", context["step_started"] - started_monotonic)\n                try:\n                    batch = next(iterator)\n                except StopIteration:\n                    return  # Preserve the loop\'s truncated-accumulation failure policy (PEP 479).\n                yield batch\n                micro += 1\n        def on_step(state: Any, metrics: dict) -> None:\n            nonlocal max_step_seconds\n            synchronize()\n            seconds = time.monotonic() - context["step_started"]\n            metrics["optimizer_step_seconds"] = seconds\n            context["step_metrics"] = metrics\n            validation_start = len(context.get("validation_timings", []))\n            try:\n                original_step(state, metrics)\n                context["last_metrics"] = dict(metrics)\n            finally:\n                context.pop("step_metrics", None)\n            # A newer FFT may overwrite its own timing key; preserve our independent measurement.\n            metrics["optimizer_step_seconds"] = seconds\n            times.append(seconds)\n            measured_validation = context.get("validation_timings", [])[validation_start:]\n            if measured_validation:\n                metrics["validation_seconds"] = sum(measured_validation)\n                evaluations.append(metrics["validation_seconds"])\n            max_step_seconds = max(max_step_seconds, seconds)\n            with (run_root / "logs/metrics.jsonl").open("a", encoding="utf-8") as stream:\n                stream.write(json.dumps({"phase": phase, "step": state.step, **metrics}) + "\\n")\n        kwargs.update(batches=guarded_batches(), on_step=on_step)\n        try:\n            return old_loop(**kwargs)\n        except _SavedBudgetStop:\n            # save_boundary already verified the seal. Return normally to legacy fft.train;\n            # the adapter reports budget_exhausted after it independently verifies final state.\n            budget_stopped = True\n            return state\n    fft._make_model_and_components, fft._make_loader = model_hook, loader_hook\n    fft.save_checkpoint, loop.run_loop, control.seal_checkpoint = save_hook, loop_hook, seal_hook\n    loop.evaluate = evaluate_hook\n    (run_root / "logs").mkdir(parents=True, exist_ok=True)\n    try:\n        if os.environ.get("VLADE_HUB_REPO"):\n            # Both phases end in /checkpoints; use distinct Hub groups.\n            group = os.environ.get("VLADE_HUB_RUN_ID", "")\n            if not group:\n                raise ValueError("Set VLADE_HUB_RUN_ID for Suite Hub uploads")\n            os.environ["VLADE_HUB_RUN_ID"] = group + "-" + phase\n        os.environ.update(VLADE_TRAIN_DEADLINE_UNIX="0", VLADE_DEADLINE_UNIX="0",\n                          VLADE_SAVE_RESERVE_SECONDS=str(reserve))\n        fft.train(cfg)\n        runs = sorted(cfg.output_dir.glob("run-*.json"), key=lambda p: p.stat().st_mtime_ns)\n        run = read(runs[-1])\n        if run["status"] not in {"completed", "budget_exhausted"}:\n            raise RuntimeError(f"FFT did not finish safely: {run[\'status\']}")\n        run.update(sampler="suite_balanced", validation_sampling="episode",\n                   suite_map_sha256=sha256(suite_map))\n        if budget_stopped:\n            run.update(status="budget_exhausted", budget_stop_verified=True)\n        write(runs[-1], run)\n        path = cfg.output_dir / f"step-{run[\'step\']:07d}"\n        control.verify_checkpoint(path, accumulation=cfg.grad_accumulation_steps)\n        payload = torch.load(path / "trainer_state.pt", map_location="cpu", weights_only=True)\n        if payload["loop"]["step"] != run["step"] or any(k not in payload for k in ("optimizer", "scheduler", "rng")):\n            raise ValueError("Final training state is incomplete")\n        result = {"status": "completed" if run["step"] == cfg.max_steps else "budget_exhausted",\n                  "phase": phase, "step": run["step"], "checkpoint": str(path),\n                  "elapsed_seconds": time.monotonic() - started_monotonic,\n                  "startup_seconds": context.get("startup_seconds", run.get("startup_seconds", 0)),\n                  "step_seconds": max(times[1:] or times, default=None),\n                  "first_step_seconds": times[0] if times else None, "saves": saves,\n                  "validation_seconds": max(evaluations, default=0),\n                  "fingerprint": fft._code_fingerprint(), "checkpoint_verified": True,\n                  "runtime_api": runtime_api,\n                  "gpu": run["hardware"], "sampler": "suite_balanced",\n                  "validation_sampling": "episode", "training_complete": run["step"] == cfg.max_steps}\n        plan = settings.get("budget", {})\n        rate = plan.get("measured_cu_per_second")\n        cu_start = plan.get("current_cu")\n        result.update(cu_usage_is_estimated=True,\n                      estimated_cu_used=rate * result["elapsed_seconds"] if rate else None,\n                      estimated_remaining_cu=cu_start - rate * result["elapsed_seconds"] if rate and cu_start else None,\n                      budget=plan)\n        report = run_root / ("pilot4/pilot_report.json" if phase == "pilot4" else "reports/run_summary.json")\n        write(report, result)\n        return result\n    except BaseException:\n        with (run_root / "logs/errors.log").open("a", encoding="utf-8") as stream:\n            stream.write(traceback.format_exc())\n        write(run_root / "reports/failure.json", {"status": "failed", "phase": phase,\n              "elapsed_seconds": time.monotonic() - started_monotonic, "auto_unassign_allowed": False})\n        raise\n    finally:\n        fft._make_model_and_components, fft._make_loader = old_model, old_loader\n        fft.save_checkpoint, loop.run_loop, control.seal_checkpoint = old_save, old_loop, old_seal\n        loop.evaluate = old_evaluate\n        for key, value in budget_env.items():\n            if value is None:\n                os.environ.pop(key, None)\n            else:\n                os.environ[key] = value\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("settings", type=Path)\n    args = parser.parse_args()\n    execute(read(args.settings))\n\n\nif __name__ == "__main__":\n    main()\n'
ADAPTER_SHA256 = '4f88f623b65685901286171a807b345c5148ee70597f8ec2d6be2ece1c43387e'
if PREPARE_INPUTS:
    target = CODE/'src/vlade/training/colab_suite_fft.py'
    if target.exists() and digest(target)!=ADAPTER_SHA256:raise ValueError('기존 local adapter SHA 불일치')
    if not target.exists():target.write_text(ADAPTER_SOURCE,encoding='utf-8')
    assert digest(target) == ADAPTER_SHA256
    control_write(RUN_DIR/'manifests/adapter.json',{'sha256':ADAPTER_SHA256,'embedded':True})

In [ ]:
PREP_CACHE_SOURCE = '"""Notebook-only verified-input receipts; metadata is never an authentication seal."""\nimport contextlib\nimport hashlib\nimport json\nimport os\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef canonical(value):\n    return json.dumps(value,sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()\n\n\ndef signature(path):\n    stat=Path(path).stat()\n    return dict(bytes=stat.st_size,mtime_ns=stat.st_mtime_ns,ctime_ns=stat.st_ctime_ns)\n\n\n@contextlib.contextmanager\ndef preparation_stage(name):\n    started=time.monotonic()\n    print(\'[준비 시작]\',name,flush=True)\n    try:\n        yield\n    except BaseException:\n        print(\'[준비 실패]\',name,f\'{time.monotonic()-started:.1f}s\',flush=True)\n        raise\n    else:\n        print(\'[준비 완료]\',name,f\'{time.monotonic()-started:.1f}s\',flush=True)\n\n\nclass DriveFileIdentity:\n    """Read-only API resolution rooted at the user-approved project folder.\n\n    FUSE size/mtime or a folder modifiedTime alone cannot identify remote content.\n    No credentials are written to receipts. An unavailable API falls back to hashing.\n    """\n    def __init__(self,project_root,root_id):\n        self.project_root=Path(project_root)\n        self.root_id=root_id\n        self.folders={():root_id}\n        self.disabled=False\n        try:\n            import google.auth\n            from google.auth.transport.requests import AuthorizedSession\n            credentials,_=google.auth.default(scopes=[\'https://www.googleapis.com/auth/drive.readonly\'])\n            self.session=AuthorizedSession(credentials)\n        except Exception as error:\n            self.disabled=True\n            print(\'Drive API 사용 불가; 원격 파일 해시 생략 불가:\',type(error).__name__,flush=True)\n\n    def __call__(self,path):\n        if self.disabled:return None\n        try:\n            parts=Path(path).relative_to(self.project_root).parts\n            if not parts or \'..\' in parts:return None\n            parent=self.root_id\n            for index,name in enumerate(parts):\n                key=parts[:index+1]\n                if index<len(parts)-1 and key in self.folders:\n                    parent=self.folders[key];continue\n                escaped=name.replace(\'\\\\\',\'\\\\\\\\\').replace("\'","\\\\\'")\n                result=self.session.get(\'https://www.googleapis.com/drive/v3/files\',params={\n                    \'q\':f"\'{parent}\' in parents and name = \'{escaped}\' and trashed = false",\n                    \'fields\':\'files(id,name,mimeType),nextPageToken\',\'pageSize\':2,\n                    \'supportsAllDrives\':\'true\',\'includeItemsFromAllDrives\':\'true\'},timeout=30)\n                result.raise_for_status();body=result.json();files=body.get(\'files\',[])\n                if len(files)!=1 or body.get(\'nextPageToken\'):return None\n                parent=files[0][\'id\']\n                if index<len(parts)-1:\n                    if files[0][\'mimeType\']!=\'application/vnd.google-apps.folder\':return None\n                    self.folders[key]=parent\n            result=self.session.get(\'https://www.googleapis.com/drive/v3/files/\'+parent,params={\n                \'fields\':\'id,name,size,version,sha256Checksum,trashed\',\'supportsAllDrives\':\'true\'},timeout=30)\n            result.raise_for_status();body=result.json()\n            if (body.get(\'trashed\') or not body.get(\'version\') or \'size\' not in body\n                    or body.get(\'name\')!=parts[-1]):return None\n            return {key:body.get(key) for key in (\'id\',\'size\',\'version\',\'sha256Checksum\')}\n        except Exception as error:\n            self.disabled=True\n            print(\'Drive 동일성 조회 실패; 이후 원격 SHA 전체 검사:\',type(error).__name__,flush=True)\n            return None\n\n\nclass VerifiedInputCache:\n    def __init__(self,receipts,local_receipts,policy,metadata=None,session_id=None):\n        self.receipts=Path(receipts);self.local_receipts=Path(local_receipts)\n        self.policy=hashlib.sha256(canonical(policy)).hexdigest()\n        self.expected_headers=policy.get(\'expected_headers\')\n        self.metadata=metadata or (lambda path:None)\n        boot=Path(\'/proc/sys/kernel/random/boot_id\')\n        self.session_id=session_id or (boot.read_text().strip() if boot.is_file() else uuid.uuid4().hex)\n\n    def load(self,path):\n        try:\n            envelope=json.loads(Path(path).read_text())\n            payload=envelope[\'payload\']\n            if (envelope[\'sha256\']!=hashlib.sha256(canonical(payload)).hexdigest()\n                    or payload[\'status\']!=\'complete\' or payload[\'policy\']!=self.policy):return None\n            return payload\n        except (ValueError,OSError,KeyError,TypeError):return None\n\n    def save(self,path,**payload):\n        path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n        payload.update(status=\'complete\',policy=self.policy)\n        envelope=dict(payload=payload,sha256=hashlib.sha256(canonical(payload)).hexdigest())\n        temporary=path.with_name(\'.\'+path.name+\'.\'+uuid.uuid4().hex+\'.incomplete\')\n        try:\n            with temporary.open(\'w\') as stream:\n                json.dump(envelope,stream,allow_nan=False);stream.flush();os.fsync(stream.fileno())\n            temporary.replace(path)\n            if self.load(path)!=payload:raise IOError(\'검증 기록 readback 실패\')\n        finally:\n            if temporary.exists():temporary.unlink()\n        return payload\n\n    def hash_file(self,path,expected=None,expected_bytes=None):\n        path=Path(path)\n        before=signature(path)\n        if expected_bytes is not None and before[\'bytes\']!=expected_bytes:raise ValueError(\'파일 크기 불일치: \'+str(path))\n        remote=self.metadata(path)\n        if remote and int(remote[\'size\'])!=before[\'bytes\']:raise ValueError(\'Drive/API 파일 크기 불일치\')\n        key=hashlib.sha256(str(path).encode()).hexdigest()\n        receipt=self.receipts/(\'file-\'+key+\'.json\')\n        previous=self.load(receipt)\n        if (previous and remote and previous.get(\'provider\')==remote\n                and previous.get(\'bytes\')==before[\'bytes\'] and previous.get(\'expected\')==expected\n                and (not remote.get(\'sha256Checksum\') or remote[\'sha256Checksum\']==previous.get(\'sha256\'))):\n            if self.metadata(path)==remote and signature(path)==before:\n                print(\'[SHA 재사용] Drive ID/서버 version 일치:\',path.name,flush=True)\n                return {**previous,\'mode\':\'provider_revision_reused\',\'stat\':before}\n        with preparation_stage(\'SHA256 \'+path.name):\n            digest=hashlib.sha256();size=0;next_report=256*1024*1024\n            with path.open(\'rb\') as stream:\n                for block in iter(lambda:stream.read(8*1024*1024),b\'\'):\n                    digest.update(block);size+=len(block)\n                    if size>=next_report:\n                        print(f\'  {path.name}: {size:,}/{before["bytes"]:,} bytes\',flush=True)\n                        next_report=size+256*1024*1024\n            actual=digest.hexdigest()\n            after_remote=self.metadata(path)\n            if signature(path)!=before or (remote and after_remote!=remote):raise ValueError(\'SHA 검사 중 파일이 변경됨\')\n            if expected is not None and actual!=expected:raise ValueError(\'SHA256 불일치: \'+path.name)\n            if remote and remote.get(\'sha256Checksum\') and remote[\'sha256Checksum\']!=actual:\n                raise ValueError(\'Drive checksum과 실제 파일 불일치\')\n            result=self.save(receipt,source_path=str(path),bytes=size,sha256=actual,expected=expected,provider=remote)\n            return {**result,\'mode\':\'full_sha256\',\'stat\':before}\n\n    def tree(self,root):\n        root=Path(root)\n        if root.is_symlink() or not root.is_dir() or not (root/\'COMPLETE\').is_file():\n            raise ValueError(\'불완전한 로컬 데이터는 재사용하지 않습니다\')\n        digest=hashlib.sha256();files=0;size=0\n        with preparation_stage(\'로컬 파일 목록/크기/mtime/ctime 확인 (내용 SHA 아님)\'):\n            for parent,dirs,names in os.walk(root):\n                dirs.sort()\n                for name in dirs:\n                    if (Path(parent)/name).is_symlink():raise ValueError(\'데이터 symlink 거부\')\n                for name in sorted(names):\n                    path=Path(parent)/name\n                    if path.is_symlink():raise ValueError(\'데이터 symlink 거부\')\n                    stat=signature(path);files+=1;size+=stat[\'bytes\']\n                    digest.update(canonical([path.relative_to(root).as_posix(),stat]));digest.update(b\'\\n\')\n                    if files%10000==0:print(f\'  파일 메타데이터 {files:,}개 확인\',flush=True)\n        return dict(files=files,bytes=size,metadata_sha256=digest.hexdigest())\n\n    def headers(self,root,info):\n        root=Path(root)\n        if not (root/\'COMPLETE\').is_file():raise ValueError(\'COMPLETE 없는 데이터\')\n        expected_headers=self.expected_headers or info[\'files_sha256\']\n        if set(expected_headers)!={\'manifest.json\',\'splits.json\',\'normalization.json\'}:\n            raise ValueError(\'필수 canonical header 목록 불일치\')\n        if expected_headers!=info[\'files_sha256\'] or not info.get(\'payloads_verified\'):\n            raise ValueError(\'고정 canonical identity/전체 검증 기록 불일치\')\n        for name,expected in expected_headers.items():\n            if name not in {\'manifest.json\',\'splits.json\',\'normalization.json\'}:raise ValueError(\'잘못된 header 목록\')\n            if hashlib.sha256((root/name).read_bytes()).hexdigest()!=expected:raise ValueError(\'데이터 header 불일치\')\n\n    def dataset(self,archive,container,suite_map,verify,extract,required_root=None,\n                expected_archive_sha=None,expected_archive_bytes=None):\n        archive=Path(archive);container=Path(container)\n        # When no archive bytes are used, the previously verified local tree is\n        # independently reusable. This does not claim the current TAR was revalidated.\n        index_key=hashlib.sha256(canonical([str(archive),self.policy])).hexdigest()\n        index_path=self.local_receipts/(\'local-index-\'+index_key+\'.json\')\n        indexed=self.load(index_path)\n        if indexed and indexed.get(\'session\')==self.session_id:\n            root=Path(indexed[\'root\'])\n            if (root.exists() and (required_root is None or str(root)==str(required_root))\n                    and (expected_archive_bytes is None or indexed[\'identity\'][\'bytes\']==expected_archive_bytes)\n                    and (expected_archive_sha is None or indexed[\'identity\'][\'sha256\']==expected_archive_sha)):\n                tree=self.tree(root)\n                if tree==indexed[\'tree\']:\n                    self.headers(root,indexed[\'info\'])\n                    print(\'[재사용] 동일 런타임의 검증된 로컬 데이터; 사용하지 않는 TAR 내용 SHA 생략\',flush=True)\n                    return root,{**indexed[\'info\'],\'verification_mode\':\'local_metadata_reuse\'},dict(\n                        indexed[\'identity\'],mode=\'archive_not_read_local_dataset_reused\',current_archive_revalidated=False)\n        identity=self.hash_file(archive,expected_archive_sha,expected_archive_bytes)\n        key=hashlib.sha256(canonical([identity[\'sha256\'],self.policy])).hexdigest()\n        receipt=self.receipts/(\'dataset-\'+key+\'.json\')\n        local_path=self.local_receipts/(\'dataset-\'+key+\'.json\')\n        previous=self.load(receipt);local=self.load(local_path)\n        root=Path(local[\'root\']) if local else None\n        if root and (required_root is None or str(root)==str(required_root)) and root.exists():\n            current=self.tree(root)\n            if local.get(\'session\')==self.session_id and current==local.get(\'tree\'):\n                self.headers(root,local[\'info\'])\n                print(\'[Episode SHA 재사용] 동일 런타임·모든 로컬 파일 메타데이터 일치\',flush=True)\n                return root,{**local[\'info\'],\'verification_mode\':\'local_metadata_reuse\'},identity\n            with preparation_stage(\'로컬 데이터 변경/재시작: 전체 Episode 검증\'):\n                info=verify(root)\n            info={**info,\'verification_mode\':\'full_payload_sha256\'}\n            self.headers(root,info);tree=self.tree(root)\n        else:\n            # Drive\'s already verified TAR is the reusable dataset. Unreceipted shared\n            # extracted directories are never trusted or copied over existing data.\n            print(\'[Drive 재사용 후보]\',archive,\'이전 전체 검증 기록:\',bool(previous),flush=True)\n            if container.exists():raise FileExistsError(\'검증되지 않은/부분 local container가 이미 존재합니다: \'+str(container))\n            container.parent.mkdir(parents=True,exist_ok=True)\n            stage=container.with_name(container.name+\'.\'+uuid.uuid4().hex+\'.incomplete\')\n            with preparation_stage(\'TAR → 새 Colab 로컬 디렉터리 압축 해제\'):\n                extracted=Path(extract(archive,stage));relative=extracted.relative_to(stage)\n            tree=self.tree(extracted)\n            remote_after=self.metadata(archive)\n            strong_source=bool(identity.get(\'provider\') and remote_after==identity[\'provider\']\n                and signature(archive)==identity[\'stat\'])\n            if previous and strong_source:\n                self.headers(extracted,previous[\'info\'])\n                if any(tree[key]!=previous[\'tree\'][key] for key in (\'files\',\'bytes\')):\n                    raise ValueError(\'재압축 해제 파일 수/전체 크기 불일치\')\n                info={**previous[\'info\'],\'verification_mode\':\'verified_drive_archive_revision_reuse\'}\n                print(\'[Episode SHA 재사용] 검증된 동일 Drive TAR revision에서 다시 압축 해제\',flush=True)\n            else:\n                with preparation_stage(\'최초/동일성 증거 부족: 전체 Episode 14,347개 payload SHA 검사\'):\n                    info=verify(extracted)\n                self.headers(extracted,info)\n                info={**info,\'verification_mode\':\'full_payload_sha256\'}\n            # COMPLETE is necessary but only this successful verification publishes a receipt.\n            if required_root is not None and str(container/relative)!=str(required_root):\n                required=Path(required_root)\n                if not relative.parts or required.parts[-len(relative.parts):]!=relative.parts:\n                    raise ValueError(\'Resume prepared_data_root와 TAR 내부 경로가 다릅니다\')\n                container=required.parents[len(relative.parts)-1]\n                if container.exists():raise FileExistsError(\'기존 Resume local 데이터는 덮어쓰지 않습니다\')\n                container.parent.mkdir(parents=True,exist_ok=True)\n            stage.rename(container);root=container/relative\n        if not info.get(\'payloads_verified\'):raise ValueError(\'전체 검증 성공 증거 없음\')\n        # Do not preserve the fast-path mode as a fresh full-verification claim.\n        if info.get(\'verification_mode\')==\'full_payload_sha256\':info[\'first_verified_at_unix\']=time.time()\n        self.save(receipt,archive_sha256=identity[\'sha256\'],info=info,tree=tree)\n        self.save(local_path,root=str(root),session=self.session_id,archive_sha256=identity[\'sha256\'],info=info,tree=tree)\n        self.save(index_path,root=str(root),session=self.session_id,identity=identity,info=info,tree=tree)\n        return root,info,identity\n'
PREP_CACHE_SHA256 = '4b44799e91dd6e3fd07c1bd2dbb855042e68e9e3530c5feb2483db596c184206'
PREP_WORKER_SOURCE = "from pathlib import Path\nimport sys,tarfile\nimport vlade.training.colab_suite_fft as adapter\nmode,source,other,report=sys.argv[1:]\nif mode=='extract':\n    original=tarfile.TarFile.extractall\n    def extract_progress(self,path='.',members=None,**kwargs):\n        selected=self.getmembers() if members is None else members\n        def progress():\n            for index,member in enumerate(selected,1):\n                yield member\n                if index%10000==0:print('  TAR 추출',index,'entries',flush=True)\n        return original(self,path,members=progress(),**kwargs)\n    tarfile.TarFile.extractall=extract_progress\n    root=adapter.extract_archive(Path(source),None,Path(other))\n    adapter.write(Path(report),{'prepared':str(root)})\nelif mode=='verify':\n    original=adapter.sha256\n    checked=[0]\n    def sha_progress(path):\n        result=original(path);checked[0]+=1\n        if checked[0]%10000==0:print('  payload SHA 확인',checked[0],'files',flush=True)\n        return result\n    adapter.sha256=sha_progress\n    info=adapter.verify_data(Path(source),Path(other))\n    adapter.write(Path(report),info)\nelse:raise ValueError('Unknown preparation action')\n"
assert hashlib.sha256(PREP_CACHE_SOURCE.encode()).hexdigest()==PREP_CACHE_SHA256
exec(compile(PREP_CACHE_SOURCE,'<verified-input-cache>','exec'),globals())


## AVA Python 3.11 격리 환경
첨부 노트북의 고정 Transformers fork/patch/wheel SHA 검사를 재사용합니다.
패키지는 Colab 로컬에만 설치하고 공유 upstream에는 쓰지 않습니다.
설치 도구를 먼저 확인하며 실패 시 실제 stdout/stderr를 출력합니다.

In [ ]:
AVA_ENV_READY = False
if PREPARE_INPUTS:
    UV = shutil.which('uv')
    if UV is None:
        # 현재 Python은 설치 도구 bootstrap에만 사용; AVA 실행에는 사용하지 않습니다.
        bootstrap = LOCAL/'uv-tool'
        run([sys.executable,'-m','venv',bootstrap])
        tool_python = bootstrap/'bin/python'
        probe = subprocess.run([str(tool_python),'-m','pip','--version'],capture_output=True,text=True)
        if probe.returncode:
            print(probe.stdout,probe.stderr)
            run([tool_python,'-m','ensurepip','--upgrade'])
        run([tool_python,'-m','pip','install','uv'])
        UV = str(bootstrap/'bin/uv')
    run([UV,'--version'])
    def environment(name, version, packages):
        directory = Path('/content')/name
        if not (directory/'bin/python').is_file():run([UV,'venv','--python',version,directory])
        python = directory/'bin/python'
        run([UV,'pip','install','--python',python,*packages])
        return python

In [ ]:
if PREPARE_INPUTS:
    AVA_ENV_READY = False
    AVA_PY = None
    import tempfile, tomllib, zipfile
    AVA_VENV = Path('/content') / ('vlade_ava_fft_py311_' + RUN_ID)
    AVA_PATCH_REL = 'configs/models/teacher/transformers-avavla-a03eee5-checkpoint-extra-attn.patch'
    AVA_PATCH_SHA = '9f0e46c9e93d62a92af950aeee712191cfcf314c1e636a9608fddcb57ff6087d'
    AVA_TRANSFORMERS_REV = 'a03eee5da1794c45f5a911b60a9a4545e6e0ed1f'
    AVA_BASE_SHA = '62f22856fe7b8459d6d61d4676b0075b65724979f84859ac475dbe36c10a3d80'
    AVA_INSTALLED_SHA = 'aeca129d863bec439623f54d93810792d82c86921c25fbe69d9169ffdb373788'
    AVA_VERIFY_SOURCE = "import ast, hashlib, importlib, json, sys\nfrom pathlib import Path\nexpected_sha, expected_prefix = sys.argv[1:3]\nprefix = Path(sys.prefix).resolve()\nif sys.prefix == sys.base_prefix or prefix != Path(expected_prefix).resolve():\n    raise RuntimeError('AVA verification requires the dedicated virtual environment')\nmodule = importlib.import_module('transformers.models.llama.modeling_llama')\nsource = Path(module.__file__).resolve()\nif not source.is_relative_to(prefix) or source.name != 'modeling_llama.py':\n    raise RuntimeError('Transformers was imported outside the AVA environment')\npayload = source.read_bytes()\nif hashlib.sha256(payload).hexdigest() != expected_sha:\n    raise RuntimeError('Installed AVA Transformers source SHA mismatch')\ntree = ast.parse(payload)\nmodel = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == 'LlamaModel')\nforward = next(node for node in model.body if isinstance(node, ast.FunctionDef) and node.name == 'forward')\ncalls = [node for node in ast.walk(forward) if isinstance(node, ast.Call)\n         and isinstance(node.func, ast.Attribute) and node.func.attr == '_gradient_checkpointing_func']\nexpected = 'extra_attn_weights if extra_layer_ids and layer_idx in extra_layer_ids else None'\nif len(calls) != 1 or len(calls[0].args) != 9 or ast.unparse(calls[0].args[-1]) != expected:\n    raise RuntimeError('Installed checkpoint attention argument was not preserved')\nprint(json.dumps({'source': str(source), 'sha256': expected_sha, 'venv': str(prefix)}))"
    _VLADE_AVA_BASE_RUN = getattr(run, '_vlade_base_run', run)

    def verify_ava_transformers(python):
        _VLADE_AVA_BASE_RUN([python, '-c', AVA_VERIFY_SOURCE, AVA_INSTALLED_SHA, AVA_VENV])

    def require_ava_environment():
        if not AVA_ENV_READY or AVA_PY is None:
            raise RuntimeError('AVA patched environment is not verified; rerun the installation cell')
        verify_ava_transformers(AVA_PY)

    def run(args, env=None):
        command = [str(value) for value in args]
        if command and (command[0] == str(AVA_VENV / 'bin/python')
                        or any(Path(value).name == 'run_fft.py' for value in command)):
            require_ava_environment()
        return _VLADE_AVA_BASE_RUN(args, env=env)

    run._vlade_base_run = _VLADE_AVA_BASE_RUN
    patch = CODE / AVA_PATCH_REL
    source_manifest = json.loads((CODE / 'experiments/manifests/colab_source.json').read_text())
    if source_manifest['files_sha256'].get(AVA_PATCH_REL) != AVA_PATCH_SHA:
        raise RuntimeError('Staged source manifest does not pin the approved AVA patch')
    if not patch.is_file() or hashlib.sha256(patch.read_bytes()).hexdigest() != AVA_PATCH_SHA:
        raise RuntimeError('Staged AVA patch is missing or has the wrong SHA')

    build_root = Path(tempfile.mkdtemp(prefix='vlade-ava-transformers-', dir=LOCAL))
    source = build_root / 'source'
    source.mkdir()
    build_env = ENV.copy()
    build_env.update(UV_CACHE_DIR=str(build_root / 'uv-cache'), TMPDIR=str(build_root / 'tmp'))
    (build_root / 'tmp').mkdir()
    fork = 'https://github.com/LiAuto-DSR/transformers-avavla.git'
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'init'], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'fetch', '--depth', '1', fork, AVA_TRANSFORMERS_REV], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'checkout', '--detach', 'FETCH_HEAD'], env=build_env)
    revision = subprocess.check_output(['git', '-c', 'core.autocrlf=false', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
    if revision != AVA_TRANSFORMERS_REV:
        raise RuntimeError('AVA Transformers checkout revision mismatch')
    llama = source / 'src/transformers/models/llama/modeling_llama.py'
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_BASE_SHA:
        raise RuntimeError('Unpatched AVA Transformers source SHA mismatch')
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'apply', '--check', patch], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'apply', patch], env=build_env)
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Patched AVA Transformers source SHA mismatch')
    wheels = build_root / 'wheels'
    wheels.mkdir()
    _VLADE_AVA_BASE_RUN([UV, 'build', '--wheel', '--python', '3.11', '--out-dir', wheels, source], env=build_env)
    wheel_files = list(wheels.glob('transformers-*.whl'))
    if len(wheel_files) != 1:
        raise RuntimeError('Expected exactly one patched Transformers wheel')
    wheel = wheel_files[0]
    with zipfile.ZipFile(wheel) as archive:
        wheel_source = archive.read('transformers/models/llama/modeling_llama.py')
    if hashlib.sha256(wheel_source).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Built Transformers wheel does not contain the approved patch')

    upstream = CODE / 'third_party/ava_vla'
    dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
    dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
    dependencies += [
        str(wheel),
        'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
        'protobuf==4.25.3', 'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
        'huggingface_hub==0.36.2', 'accelerate==1.15.0']
    candidate_python = environment(('vlade_ava_fft_py311_' + RUN_ID), '3.11', dependencies)
    if str(candidate_python) != str(AVA_VENV / 'bin/python'):
        raise RuntimeError('Refusing to install outside the dedicated AVA environment')
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python,
                        '--no-deps', '--force-reinstall', wheel], env=build_env)
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python, '--no-deps', '-e', upstream])
    verify_ava_transformers(candidate_python)
    _VLADE_AVA_BASE_RUN([UV,'pip','check','--python',candidate_python])
    cpu_import_env = ENV.copy()
    cpu_import_env['CUDA_VISIBLE_DEVICES'] = ''
    _VLADE_AVA_BASE_RUN([candidate_python,'-c',
        "import sys,importlib,json;assert sys.version_info[:2]==(3,11);names=['torch','numpy','transformers','accelerate','peft','bitsandbytes','draccus','dlimp','tensorflow','tensorflow_datasets','tensorflow_graphics','tensorflow_metadata','google.protobuf'];[importlib.import_module(n) for n in names];print(json.dumps(dict(cpu_imports='passed',modules=names,python=sys.version)))"],env=cpu_import_env)
    _VLADE_AVA_BASE_RUN([candidate_python, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
    freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(candidate_python)], text=True)
    (RUN_DIR / 'manifests/ava_environment_freeze.txt').write_text(freeze)
    (AVA_VENV / 'vlade_transformers_patch.json').write_text(json.dumps({
        'base_commit': AVA_TRANSFORMERS_REV, 'patch_sha256': AVA_PATCH_SHA,
        'installed_source_sha256': AVA_INSTALLED_SHA,
        'wheel_sha256': hashlib.sha256(wheel.read_bytes()).hexdigest()}, indent=2))
    AVA_PY = candidate_python
    AVA_ENV_READY = True
    print('AVA managed Transformers patch verified; FFT environment ready')
    control_write(RUN_DIR/'manifests/environment.json',{
      'python':str(AVA_PY),'patch_sha256':AVA_PATCH_SHA,'transformers_revision':AVA_TRANSFORMERS_REV,
      'installed_transformers_sha256':AVA_INSTALLED_SHA,'freeze_sha256':digest(RUN_DIR/'manifests/ava_environment_freeze.txt')})
else:
    print('Python 환경 준비 대기; Colab 기본 Python으로 AVA를 실행하지 않습니다.')

## 로컬 데이터 전체 무결성 및 실제 Processor/Collator
아래 셀은 학습을 실행하지 않습니다. 최초에는 TAR SHA·로컬 압축 해제·전체 payload SHA 검사를 수행합니다. 이후에는 파일 동일성 근거를 확인한 검증 기록을 재사용하고 사용한 검증 모드를 기록합니다.
고정 prepared manifest SHA는 이번 작업에서 기존 로컬 canonical D 보관본을 읽어 기록한 값입니다.
이 값의 신뢰 근거가 불충분하거나 파일이 다르면 변경해서 우회하지 말고 검증된 준비 기록을 확인하세요.
T0 weight/component는 기존 download manifest의 해시를 확인합니다.
공유 checkpoint의 변경 이력이 있는 config/Python 파일은 로컬에 읽기 전용으로 복사하고 실제 SHA를 기록합니다.
모델 자체의 로딩·forward/backward·재로딩 성공은 파일럿에서 별도로 확인합니다.

In [ ]:
if PREPARE_INPUTS:
    require_ava_environment()
    run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.colab_suite_fft import runtime_api_report,write;write(Path(sys.argv[2]),runtime_api_report(Path(sys.argv[1])))",CODE,RUN_DIR/'manifests/runtime_api.json'])
    print((RUN_DIR/'manifests/runtime_api.json').read_text())
    from vlade.storage.assets import asset_record
    PREPARED=hf_input('prepared_d_seed42',AVA_PY)
    T0=hf_input('ava_t0',AVA_PY)
    map_path=PREPARED/'suite_map.json'
    if not map_path.is_file():
        map_root=hf_input(SUITE_MAP_ASSET,AVA_PY)
        map_path=map_root/'suite_map.json'
    if not map_path.is_file(): raise FileNotFoundError('Publish the original audited suite_map.json to HF first; labels are never inferred.')
    SUITE_MAP=LOCAL/'suite_map.json'
    if SUITE_MAP.exists() and digest(SUITE_MAP)!=digest(map_path):raise ValueError('Resume suite map changed')
    if not SUITE_MAP.exists():shutil.copyfile(map_path,SUITE_MAP)
    prep_script=LOCAL/'prepare_inputs.py'
    prep_script.write_text(PREP_WORKER_SOURCE)
    data_report=RUN_DIR/'manifests/dataset_full_validation.json'
    run([AVA_PY,prep_script,'verify',PREPARED,SUITE_MAP,data_report])
    DATA_INFO=json.loads(data_report.read_text())
    DATA_INFO['verification_mode']='pinned_hf_archive_and_full_payload_validation'
    control_write(LOCAL/'prepared.json',{'prepared':str(PREPARED)})
    control_write(RUN_DIR/'manifests/dataset_manifest.json',DATA_INFO)
    control_write(RUN_DIR/'manifests/hf_inputs.json',{
        'prepared':asset_record('prepared_d_seed42'),'teacher':asset_record('ava_t0')})
    config = json.loads((CODE/'configs/experiments/libero_plus_fft.json').read_text())
    total_steps = math.ceil(228908*MAX_EPOCHS/(BATCH_SIZE*GRAD_ACCUMULATION_STEPS))
    config['fft_defaults'].update(prepared_data_root=str(PREPARED),data_root=str(PREPARED),
       image_aug=False,batch_size=BATCH_SIZE,grad_accumulation_steps=GRAD_ACCUMULATION_STEPS,
       seed=42,max_steps=total_steps,save_steps=SAVE_STEPS,save_optimizer_state=SAVE_OPTIMIZER_STATE,
       eval_steps=SAVE_STEPS,max_eval_batches=16,log_steps=1,debug_records=0,
       lr_schedule='constant',warmup_steps=0)
    config['fft_runs']['ava_vla'].update(checkpoint=str(T0),output_dir=str(RUN_DIR/'checkpoints'))
    MAIN_CONFIG = RUN_DIR/'configs/effective_config.json'
    if MAIN_CONFIG.exists() and json.loads(MAIN_CONFIG.read_text()) != config:
        raise ValueError('기존 본 학습 설정과 다릅니다. 덮어쓰지 않습니다.')
    control_write(MAIN_CONFIG,config)
    pilot_config = json.loads(json.dumps(config))
    pilot_config['fft_defaults'].update(max_steps=4,save_steps=4,eval_steps=4)
    pilot_config['fft_runs']['ava_vla']['output_dir']=str(RUN_DIR/'pilot4/checkpoints')
    PILOT_CONFIG = RUN_DIR/'configs/pilot4.json'
    if PILOT_CONFIG.exists() and json.loads(PILOT_CONFIG.read_text()) != pilot_config: raise ValueError('기존 pilot 설정 충돌')
    control_write(PILOT_CONFIG,pilot_config)
    inspect_script = LOCAL/'inspect_batch.py'
    inspect_script.write_text("""
import sys, torch
from pathlib import Path
from vlade.training import fft
from vlade.training.colab_suite_fft import install_hooks
from vlade.training.run_control import configure_tensorflow_cpu
configure_tensorflow_cpu(42)
fft._activate_upstream('ava_vla')
install_hooks(Path(sys.argv[2]))
cfg=fft.FFTConfig.from_json(Path(sys.argv[1]),'ava_vla')
fft.preflight(cfg)
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor
AutoConfig.register('openvla',OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig,PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig,PrismaticProcessor)
p=AutoProcessor.from_pretrained(cfg.checkpoint,trust_remote_code=True,local_files_only=True)
for train in (True,False):
    batch=next(iter(fft._make_loader(cfg,None,p,train=train)[0]))
    assert batch['actions'].shape[-2:]==(8,7)
    assert batch['proprio'].shape[-1]==8 and torch.isfinite(batch['actions']).all()
    assert batch['action_valid_mask'].shape[-2:]==(8,7)
    print('Actual processor/collator:', 'train suite_balanced' if train else 'validation episode',
          {k:list(v.shape) for k,v in batch.items() if hasattr(v,'shape')})
""")
    run([AVA_PY,inspect_script,MAIN_CONFIG,SUITE_MAP])
    READY=True
    print('HF inputs, full dataset validation and actual CPU collator verified. Training has not started.')


## 4-step 파일럿 · 저장과 재로딩
Colab UI에서 직전 CU를 읽어 `CU_BEFORE_PILOT`에 입력하고 RUN_PILOT=True로 실행합니다.
상한과 Drive 실제 여유 공간도 필요합니다. 모델·8-bit Optimizer 상태 저장 예상량은 T0 크기의 4배로
보수적으로 계산하고 파일럿 후 실제 checkpoint 크기로 대체합니다. 파일럿 가중치는 본 학습에 자동 사용되지 않습니다.

In [ ]:
def drive_free():
    return shutil.disk_usage(OUTPUT_ROOT).free

def launch(settings, name):
    path=RUN_DIR/'configs'/name
    control_write(path,settings)
    run([AVA_PY,'-m','vlade.training.colab_suite_fft',path])
def reload_check(checkpoint, config_path, report):
    script=LOCAL/'reload_check.py'
    script.write_text("""
import sys, runpy
from pathlib import Path
from vlade.training.colab_suite_fft import install_hooks
install_hooks(Path(sys.argv[1]))
sys.argv=['check_fft_checkpoint.py',*sys.argv[2:]]
runpy.run_path('scripts/train/check_fft_checkpoint.py',run_name='__main__')
""")
    run([AVA_PY,script,SUITE_MAP,'--checkpoint',checkpoint,'--data',PREPARED,
         '--config',config_path,'--report',report])
if RUN_PILOT:
    if not READY: raise RuntimeError('먼저 준비 및 실제 batch 검사를 완료하세요.')
    require_ava_environment()
    if CU_BEFORE_PILOT is None or CU_BEFORE_PILOT<=TARGET_REMAINING_CU:
        raise ValueError('파일럿 직전의 실제 CU를 입력하세요.')
    if PILOT_TIME_LIMIT_SECONDS is None or PILOT_TIME_LIMIT_SECONDS<=600:
        raise ValueError('파일럿 로딩·4-step·저장·재로딩을 포함하는 명시적 시간 상한(600초 초과)이 필요합니다.')
    checkpoint_estimate = 4*sum(p.stat().st_size for p in T0.iterdir() if p.suffix in {'.pt','.safetensors'})
    if drive_free()<checkpoint_estimate*1.35: raise ValueError('파일럿 checkpoint 저장 공간 부족')
    PILOT_STARTED_UNIX=time.time()
    pilot_deadline=PILOT_STARTED_UNIX+PILOT_TIME_LIMIT_SECONDS
    if SESSION_DEADLINE_UNIX is not None: pilot_deadline=min(pilot_deadline,SESSION_DEADLINE_UNIX)
    launch(dict(authorized=True,phase='pilot4',config=str(PILOT_CONFIG),run_root=str(RUN_DIR),
       suite_map=str(SUITE_MAP),deadline_unix=pilot_deadline,reserve_seconds=300,stop_step=4,
       free_bytes_at_start=DRIVE_FREE_BYTES_AT_START,checkpoint_bytes=checkpoint_estimate),'pilot4_launch.json')
    pilot=json.loads((RUN_DIR/'pilot4/pilot_report.json').read_text())
    if pilot['step']!=4 or not pilot['training_complete'] or pilot['validation_seconds']<=0:
        raise RuntimeError('파일럿 4-step 또는 Validation 미완료; 본 학습 금지')
    before=time.time()
    reload_check(Path(pilot['checkpoint']),PILOT_CONFIG,RUN_DIR/'pilot4/reload_report.json')
    pilot['reload_seconds']=time.time()-before
    pilot['measurement_started_unix']=PILOT_STARTED_UNIX
    pilot['measurement_ended_unix']=time.time()
    pilot['cu_before']=CU_BEFORE_PILOT
    control_write(RUN_DIR/'pilot4/pilot_report.json',pilot)
    print('파일럿 완료. Colab UI의 종료 후 CU를 CU_AFTER_PILOT에 입력하세요.')
else:
    print('파일럿 미실행: RUN_PILOT=False')

## 파일럿 기반 예산 · 본 학습 명시적 실행
이 셀 직전에 `CU_AFTER_PILOT`과 `CURRENT_COMPUTE_UNITS`를 갱신하세요.
CU 측정 구간은 파일럿 직전부터 저장/재로딩 종료 직후까지입니다. 세션/수동 상한은 계획 시점의 잔여량입니다.
전체 목표 Step은 고정하여 cosine/attention 스케줄과 Resume 의미를 바꾸지 않습니다.
세션에서 가능한 Step은 별도로 중단합니다. 2000 미만이어도 마지막 checkpoint를 저장합니다.
재개는 명시한 동일 Run의 `checkpoints/step-XXXXXXX`만 허용합니다.

In [ ]:
BUDGET=None
if READY and (RUN_DIR/'pilot4/pilot_report.json').is_file():
    require_ava_environment()
    pilot=json.loads((RUN_DIR/'pilot4/pilot_report.json').read_text())
    reload_report=json.loads((RUN_DIR/'pilot4/reload_report.json').read_text())
    if reload_report['status']!='passed' or pilot['step']!=4: raise ValueError('파일럿 재로딩 증거 없음')
    probe=LOCAL/'pilot_context_check.json'
    run([AVA_PY,'-c',"import sys,torch;from pathlib import Path;from vlade.training.colab_suite_fft import install_hooks,write;write(Path(sys.argv[2]),dict(fingerprint=install_hooks(Path(sys.argv[1]))['fingerprint'],gpu=dict(gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)))",SUITE_MAP,probe])
    observed=json.loads(probe.read_text())
    if observed['fingerprint']!=pilot['fingerprint'] or observed['gpu']!=pilot['gpu']:
        raise ValueError('파일럿과 현재 소스/환경/Teacher/GPU/VRAM이 다릅니다. 새 파일럿 필요')
    profile={k:pilot[k] for k in ['step_seconds','startup_seconds','reload_seconds','validation_seconds']}
    # 첫 step의 모델 warm-up/전체 Query 인덱스 준비 비용은 별도 startup으로 반영합니다.
    profile['startup_seconds']+=max(0,pilot['first_step_seconds']-profile['step_seconds'])
    profile['checkpoint_seconds']=max(x['save_verify_seconds'] for x in pilot['saves'])
    checkpoint_bytes=max(x['bytes'] for x in pilot['saves'])
    resume_step=0
    if RESUME_CHECKPOINT:
        resume=Path(RESUME_CHECKPOINT)
        if not resume.resolve().is_relative_to((RUN_DIR/'checkpoints').resolve()): raise ValueError('다른 Run/pilot 재개 금지')
        resume_step=json.loads((resume/'vlade_fft.json').read_text())['step']
    plan_args=dict(profile=profile,current_cu=CURRENT_COMPUTE_UNITS,target_cu=TARGET_REMAINING_CU,
       cu_before=pilot['cu_before'],cu_after=CU_AFTER_PILOT,
       cu_elapsed=pilot['measurement_ended_unix']-pilot['measurement_started_unix'],
       session_seconds=None if SESSION_DEADLINE_UNIX is None else SESSION_DEADLINE_UNIX-time.time(),
       explicit_seconds=EXPLICIT_MAIN_TIME_LIMIT_SECONDS,free_bytes=drive_free(),
       checkpoint_bytes=checkpoint_bytes,remaining_steps=total_steps-resume_step,save_steps=SAVE_STEPS,
       resume_step=resume_step)
    # 명시적 시간 모드 선택: CU 실측이 둘 다 없을 때만 소모율을 생략합니다.
    if CU_AFTER_PILOT is None: plan_args.update(cu_before=None,cu_after=None)
    plan_input=LOCAL/'plan_input.json'; plan_output=LOCAL/'plan_output.json'
    control_write(plan_input,plan_args)
    run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.colab_suite_fft import read,write,plan_budget;write(Path(sys.argv[2]),plan_budget(**read(Path(sys.argv[1]))))",plan_input,plan_output])
    BUDGET=json.loads(plan_output.read_text())
    control_write(RUN_DIR/'reports/budget.json',BUDGET)
    print(json.dumps(BUDGET,indent=2,ensure_ascii=False))
    if not BUDGET['cu_target_supported']: print('명시적 시간 모드: 5 CU 잔여 목표 제어 불가')
if RUN_MAIN_FFT:
    try:
        if BUDGET is None: raise RuntimeError('현재 조건의 파일럿 기반 예산이 필요합니다.')
        main_started=time.time()
        main_deadline=main_started+BUDGET['wall_seconds']
        launch(dict(authorized=True,phase='main',config=str(MAIN_CONFIG),run_root=str(RUN_DIR),
          suite_map=str(SUITE_MAP),resume=RESUME_CHECKPOINT,
          deadline_unix=main_deadline,reserve_seconds=BUDGET['save_reserve_seconds'],
          stop_step=resume_step+BUDGET['steps_this_session'],step_seconds=profile['step_seconds'],budget=BUDGET,
          first_step_seconds=pilot['first_step_seconds'],
          free_bytes_at_start=DRIVE_FREE_BYTES_AT_START,checkpoint_bytes=checkpoint_bytes),'main_launch.json')
        summary=json.loads((RUN_DIR/'reports/run_summary.json').read_text())
        if summary['status'] not in {'completed','budget_exhausted'} or not summary['checkpoint_verified']:
            raise RuntimeError('정상 저장 확인 실패; 런타임을 해제하지 않습니다.')
        # Re-read the checkpoint seal and all payloads in a separate process before unassign.
        run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.run_control import verify_checkpoint;print(verify_checkpoint(Path(sys.argv[1]))['step'])",summary['checkpoint']])
        summary['drive_readback_verified']=True
        summary['elapsed_seconds_including_final_readback']=time.time()-main_started
        rate=BUDGET['measured_cu_per_second']
        if rate is not None:
            summary['estimated_cu_used']=rate*summary['elapsed_seconds_including_final_readback']
            summary['estimated_remaining_cu']=CURRENT_COMPUTE_UNITS-summary['estimated_cu_used']
        summary['cu_estimate_excludes_unmount_flush']=True
        control_write(RUN_DIR/'reports/run_summary.json',summary)
        print(json.dumps(summary,indent=2,ensure_ascii=False))
        if AUTO_UNASSIGN_RUNTIME:
            from google.colab import runtime
            # 오류 시 unassign에 도달하지 않습니다. 공유 원본에는 쓰지 않고 이 Run의 저장을 flush합니다.
            remaining=main_deadline-time.time()
            if remaining<=0: raise RuntimeError('Drive flush 예비시간 초과; 자동 해제를 하지 않습니다.')
            drive.flush_and_unmount(timeout_ms=int(min(300,remaining)*1000))
            runtime.unassign()
    except BaseException:
        import traceback
        traceback.print_exc()
        error=traceback.format_exc()
        (LOCAL/'main_failure.log').write_text(error)
        if RUN_DIR.exists():
            with (RUN_DIR/'logs/errors.log').open('a') as stream: stream.write(error)
        raise

else:
    print('본 학습 미실행: RUN_MAIN_FFT=False. 가능한 Step/CU/용량은 파일럿 및 사용자 입력 전에는 미확정입니다.')

## 재시작 및 한계
기존 Run ID와 명시적 본 학습 checkpoint를 지정하고 로컬 준비를 다시 수행합니다.
기존 고정 설정이 다르면 중단합니다. batch/accumulation/seed/샘플링·suite map·소스·정규화·Optimizer
호환성을 검사합니다. Query cursor는 `step × accumulation × batch`를 같은 seed로 재생/skip합니다.
num_workers=0, image_aug=false를 전제로 하며 GPU의 bitwise 일치까지 보장하지 않습니다.
파일럿과 GPU/VRAM이 바뀌면 해당 환경의 새 파일럿이 필요합니다.
오류 및 저장 실패는 traceback/stderr를 남기며 런타임 자동 해제를 수행하지 않습니다.
Drive readback 후 `drive.flush_and_unmount()`가 성공한 경우에만 unassign합니다.
이는 Drive 서버의 영구 내구성을 보장한다는 의미는 아닙니다.

## 준비 검증 캐시의 범위
검증 기록은 OUTPUT_ROOT/verified_inputs에 저장합니다. 동일 런타임의 로컬 데이터는 COMPLETE, 고정 manifest 및 모든 파일의 크기·mtime·ctime 목록이 일치할 때 내용 SHA를 재사용합니다. 메타데이터 확인은 내용 해시의 재측정이 아니며, 관리자에 의한 변조까지 막는 서명이 아닙니다.
런타임이 초기화되면 Drive의 기존 TAR와 이전 전체 검증 기록을 확인합니다. 같은 경로를 Drive API에서 파일 ID와 서버 version으로 다시 확인할 수 있을 때 TAR/T0 해시와 재압축 해제된 payload의 이전 검증을 재사용합니다. SHA256Checksum이 제공되면 추가 대조합니다. API 인증·조회가 불가능하면 원격 SHA 생략을 보장하지 않으며 필요한 전체 검사를 유지합니다.
검증 기록만 있고 실제 파일 동일성이 확인되지 않으면 재사용하지 않습니다. 부분 디렉터리·COMPLETE 누락·실패/손상 기록은 거부합니다. 공유 압축 해제본은 검증 기록이 없으면 신뢰하지 않으며, 기존 TAR·데이터·Teacher는 수정하지 않습니다. 새 로컬 디렉터리와 개인 출력의 검증 기록만 작성합니다.
로컬 메타데이터는 재측정 SHA와 구분하여 기록하며, 실제 학습 시 기존 EpisodeData의 trajectory/query/image 읽기 시점 SHA 검사를 유지합니다.
